In [18]:
import pandas as pd

def read_input(file_name, lang):
    triple_df = pd.read_json('data/dataset_v2/entities_data_complete/'+file_name+'.json', orient='index')
    #sort by sitelink count
    triple_df_sorted = triple_df.sort_values(by='sitelinks',ascending=False)

    #check if entity label in all languages
    #if True, generate prompts
    #triple_df_sorted_new = triple_df_sorted[triple_df_sorted['labels_count']==8] #exclude hindi
    #print(lang,':',len(triple_df_sorted_new),':',len(triple_df_sorted))
    return triple_df_sorted

In [19]:
instruction_template = {
'ar_pob':"""##Instruction
        أجب عن السؤال باللغة العربية.
        قم بالإجابة بالكلمة المفتاحية فقط. 
        ولا تقم بإرفاق أي تفسيرات.
        أعد اسم المدينة كإجابة. 
        ##Question 
        ما هو مسقط رأس {} ؟ """,
'de_pob':"""##Instruction
            Beantworte die Frage in Deutsch.
            Gib die Antwort nur in Form eines Schlüsselworts zurück.
            Füge keine Erklärungen hinzu.
            Gib den Namen der Stadt als Antwort zurück.
            ##Question
            Was ist der Geburtsort von {}?""",
'en_pob':"""##Instruction
            Answer the question in English. 
            Return only the answer keyword. 
            Do not include any explanations. 
            Return the city name as answer.  
            ##Question 
            What is the birthplace of {}?""",
'fr_pob':"""##Instruction
            Réponds à la question en français.
            Retourne uniquement le mot-clé de la réponse.
            Aucune explication ne doit être incluse.
            Indiquez le nom de la ville comme réponse. 
            ##Question 
            Quel est le lieu de naissance de {} ?""",
'hi_pob':"""##Instruction
            प्रश्न का उत्तर हिंदी में दें।
            केवल उत्तर शब्द लौटाएँ।
            कोई स्पष्टीकरण शामिल न करें।
            उत्तर के रूप में शहर का नाम लौटाएँ।
            ##Question
            {} का जन्मस्थान क्या है?""",
'it_pob':"""##Instruction
            Rispondi in Italiano. 
            Rispondi soltanto con la risposta della domanda. 
            Non includere commenti o spiegazioni alla risposta. 
            Rispondi soltanto con il paese in cui è nata la persona.
            ##Question 
            Qual è il luogo di nascita di {}?""",
'pl_pob':"""##Instruction
            Odpowiedz na pytanie w języku polskim. 
            Podaj wyłącznie nazwę miejscowości. 
            Nie dodawaj żadnych wyjaśnień. 
            ##Question  
            Gdzie urodził/urodziła się {}?""",
'ru_pob':"""##Instruction
            Отвечайте на вопрос на Русском.
            Возвращайте только краткий ответ в виде ключевого слова.
            Не добавляйте никаких объяснений.
            Возвращайте название города в качестве ответа.
            ##Question
            Где родился/родилась {}?""",
'zh_pob':"""##Instruction
            用繁體中文回答給出的問題。
            只回答關鍵詞答案。
            不要給出任何解釋。
            返回答案必須是城市。
            ##Question
            {}的出生地是？""",
'ar_dob':"""##Instruction
            أجب عن السؤال باللغة العربية.
            قم بالإجابة بالكلمة المفتاحية فقط. 
            ولا تقم بإرفاق أي تفسيرات.
            قم بإرجاع الإجابات بصيغة ”YYYYY-MM-DD“. 
            ##Question
            ما هو تاريخ ميلاد {} ؟""",
'de_dob':"""##Instruction
            Beantworte die Frage in Deutsch.
            Gib die Antwort nur in Form eines Schlüsselworts zurück.
            Füge keine Erklärungen hinzu.
            Gib Antworten im Format ‘YYYY-MM-DD’ zurück.
            ##Question
            Was ist das Geburtsdatum von {}?""",
'en_dob':"""##Instruction
            Answer the question in English. 
            Return only the answer keyword. 
            Do not include any explanations. 
            Return answers in 'YYYY-MM-DD' format.  
            ##Question 
            What is the birth date of {}?""",
'fr_dob':"""##Instruction
            Réponds à la question en français.
            Retourne uniquement le mot-clé de la réponse.
            Aucune explication ne doit être incluse.
            La réponse doit respecter le format 'YYYY-MM-DD'. 
            ##Question 
            Quelle est la date de naissance de {}?""",
'hi_dob':"""##Instruction
            प्रश्न का उत्तर हिंदी में दें।
            केवल उत्तर शब्द लौटाएँ।
            कोई स्पष्टीकरण शामिल न करें।
            उत्तर 'YYYY-MM-DD' प्रारूप में लौटाएँ।
            ##Question
            {} की जन्म तिथि क्या है?""",
'it_dob':"""##Instruction
            Rispondi in Italiano. 
            Rispondi soltanto con la risposta della domanda. 
            Non includere commenti o spiegazioni alla risposta. 
            Ritorna la risposta nel seguente formato: 'YYYY-MM-DD'. 
            ##Question
            Qual è la data di nascita di {}?""",
'pl_dob':"""##Instruction 
            Odpowiedz na pytanie w języku polskim. 
            Odpowiedz na pytanie podając wyłącznie datę w formacie „RRRR-MM-DD". 
            ##Question 
            # Kiedy urodził/urodziła się {}?""",
'ru_dob':"""##Instruction
            Отвечайте на вопрос на Русском.
            Возвращайте только краткий ответ в виде ключевого слова.
            Не добавляйте никаких объяснений.
            Возвращайте ответы в формате ‘YYYY-MM-DD’.
            ##Question
            Какова дата рождения {}?""",
'zh_dob':"""##Instruction
            用繁體中文回答給出的問題。
            只回答關鍵詞答案。
            不要給出任何解釋。
            返回答案遵循下面的形式’YYYY-MM-DD’。
            ##Question
            {}的生日是？""",
'ar_occ':"""##Instruction
                أجب عن السؤال باللغة العربية.
                قم بالإجابة بالكلمة المفتاحية فقط. 
                الإجابة لابد أن تكون على هذا الشكل: [الإجابة 1، الإجابة 2، ...].
                ولا تقم بإرفاق أي تفسيرات.
                ##Question 
                ما هي وظيفة {} ؟""",
'de_occ':"""##Instruction
                Beantworte die Frage in Deutsch.
                Gib die Antwort nur in Form eines Schlüsselworts zurück.
                Die Antwort muss diesem Format entsprechen: [Antwort1, Antwort2, ...]
                Füge keine Erklärungen hinzu.
                ##Question
                Was ist der Beruf von {}?""",
'en_occ':"""##Instruction
                Answer the question in English. 
                Return only the answer keyword.
                Answer must follow this format: [answer1, answer2,...] 
                Do not include any explanations.  
                ##Question 
                What is the occupation of {}?""",
'fr_occ':"""##Instruction
                Réponds à la question en français.
                Retourne uniquement le mot-clé de la réponse.
                Aucune explication ne doit être incluse.
                La réponse doit respecter le format suivant : [answer1, answer2,...].
                ##Question 
                Quelle est la profession de {} ?""",
'hi_occ':"""##Instruction
                प्रश्न का उत्तर हिंदी में दें।
                केवल उत्तर शब्द  लौटाएँ।
                उत्तर का प्रारूप इस प्रकार होना चाहिए: [उत्तर 1, उत्तर 2,...]।
                कोई स्पष्टीकरण शामिल न करें।
                ##Question
                {} का व्यवसाय क्या है?""",
'it_occ':"""##Instruction
                Rispondi in Italiano. 
                Rispondi soltanto con la parola chiave richiesta. 
                La risposta deve seguire questo formato: [answer1, answer2,...]
                Non includere spiegazioni. 
                ##Question 
                Qual è la occupazione (o il lavoro) di {}?""",
'pl_occ':"""##Instruction
                Odpowiedz na pytanie w języku polskim.
                Podaj tylko odpowiedź.
                Odpowiedź musi mieć format: [odpowiedź1, odpowiedź2,...]
                Nie dodawaj żadnych wyjaśnień.
                ##Question 
                Kim z zawodu jest {}?""",
'ru_occ':"""##Instruction
                Отвечайте на вопрос на Русском.
                Возвращайте только краткий ответ в виде ключевого слова.
                Ответ должен соответствовать формату: [ответ1, ответ2, …]
                Не добавляйте никаких объяснений.
                ##Question
                Какова профессия {}?""",
'zh_occ':"""##Instruction
                用繁體中文回答給出的問題。
                只回答關鍵詞答案。
                返回答案遵循下面的形式: [‘答案1’, ‘答案2’，...]。
                不要給出任何解釋。
                ##Question
                {}的職業是？""",
'ar_ctry':"""##Instruction
                أجب عن السؤال باللغة العربية.
                قم بالإجابة بالكلمة المفتاحية فقط. 
                الإجابة لابد أن تكون على هذا الشكل: [الإجابة 1، الإجابة 2، ...].
                ولا تقم بإرفاق أي تفسيرات.
                ##Example
                س: : ما هي دولة جنسية ماري كوري؟
                تنسيق غير صحيح: فرنسية
                التنسيق الصحيح: [فرنسا]
                ##Question 
                ما هي دولة جنسية {}؟""",
'de_ctry':"""##Instruction
                Beantworte die Frage in Deutsch.
                Gib die Antwort nur in Form eines Schlüsselworts zurück.
                Die Antwort muss diesem Format entsprechen: [Antwort1, Antwort2, ...]
                Füge keine Erklärungen hinzu.
                ##Example
                Frage: Was ist die Staatsangehörigkeit von Marie Curie?
                Falsches Format: Französisch
                Richtiges Format: Frankreich
                ##Question
                Was ist die Staatsangehörigkeit von {}?""",
'en_ctry':"""##Instruction
                Answer the question in English.
                Return only the answer keyword. 
                Answer must follow this format: [answer1, answer2,...]
                Do not include any explanations.
                ##Example
                Q: What is the country of citizenship of Marie Curie?
                Incorrect format: French
                Correct format: [France]
                ##Question
                What is the country of citizenship of {}?""",
'fr_ctry':"""##Instruction
                Réponds en français.
                Retourne uniquement le ou les mots-clés de la réponse.
                La réponse doit respecter le format suivant : [réponse1, réponse2,...]
                Aucune explication ne doit être incluse.
                ##Example
                Q: Quel est le pays de citoyenneté de Marie Curie ?
                Format incorrect : française
                Format correct : [France]
                ##Question
                Quel est le pays de citoyenneté de {} ?""",
'hi_ctry':"""##Instruction
                प्रश्न का उत्तर हिंदी में दें।
                केवल उत्तर शब्द लौटाएँ।
                उत्तर का प्रारूप इस प्रकार होना चाहिए: [उत्तर 1, उत्तर 2,...]।
                कोई स्पष्टीकरण शामिल न करें।
                ##Example
                प्रश्न: मैरी क्यूरी किस देश की नागरिक हैं?
                गलत प्रारूप: फ्रेंच
                सही प्रारूप: [फ्रांस]
                ##Question
                {} की नागरिकता किस देश की है?""",
'it_ctry':"""##Instruction
                Rispondi in italiano. 
                Fornisci soltanto la parola chiave richiesta. 
                La risposta deve seguire questo formato: [risposta1, risposta2,...]
                Non includere spiegazioni.
                ##Example
                Q: Qual è lo Stato di cittadinanza di Marie Curie?
                Formato errato: francese
                Formato corretto: [Francia]
                ##Question
                Qual è lo Stato di cittadinanza di 
                {}?""",
'pl_ctry':"""##Instruction
                Odpowiedz na pytanie w języku polskim.
                Podaj tylko odpowiedź. 
                Odpowiedź musi mieć format: [odpowiedź1, odpowiedź2,...] 
                Nie dodawaj żadnych wyjaśnień. 
                ##Example
                Q: Jakiego państwa obywatelem jest Maria Skłodowska-Curie? 
                Nieprawidłowy format: francuskie 
                Prawidłowy format: [Francja] 
                ##Question 
                Jakiego państwa obywatelem jest {}?""",
'ru_ctry':"""##Instruction 
                Отвечайте на вопрос на русском языке.
                Возвращайте только краткий ответ в виде ключевого слова.
                Ответ должен соответствовать формату: [ответ1, ответ2,...]
                Не добавляйте никаких объяснений.
                ##Example
                Вопрос: Какое гражданство у Марии Кюри?
                Неправильный формат: французское
                Правильный формат: Франция
                ##Question
                Какое гражданство у {}?""",
'zh_ctry':"""##Instruction
               用繁體中文回答給出的問題。
                只回答關鍵詞答案。
                返回答案遵循下面的形式: [‘答案1’, ‘答案2’，...]。
                不要給出任何解釋。
                ##Example
                問: 瑪麗·居里的國籍是？
                錯誤答案格式: 法國的
                正確答案格式: [法國]
               ##Question
               {}的國籍是？"""
}

In [20]:
def prompt_generate_pob(ent, df, lang):
    gt = []
    prompt=''
    ent_pob = df['P19']['values']
    for val in ent_pob:
        pob = val['labels']
        if lang in pob.keys():
            gt.append(pob[lang])      
            prompt = instruction_template.get(lang+'_pob').format(ent)
    return gt, prompt
    

In [21]:
def prompt_generate_dob(ent, df, lang):
    gt, prompt='',''
    ent_dob = df['P569']['values']
    gt = ent_dob[0]['qid']
    prompt = instruction_template.get(lang+'_dob').format(ent)
    return gt, prompt

In [22]:
def prompt_generate_occ(ent, df, lang):
    gt = []
    prompt=''
    if not pd.isnull(df['P106']):
        ent_occupation = df['P106']['values']
        for job in ent_occupation:
            occupation = job['labels']
            if lang in occupation.keys():
                gt.append(occupation[lang])
                prompt = instruction_template.get(lang+'_occ').format(ent)
    return gt, prompt

In [23]:
def prompt_generate_country(ent, df, lang):
    gt = []
    prompt=''
    if not pd.isnull(df['P27']):
        ent_citizen_country = df['P27']['values']   
        for country in ent_citizen_country:
            citizen_country = country['labels']
            if lang in citizen_country.keys():
                gt.append(citizen_country[lang])
                prompt = instruction_template.get(lang+'_ctry').format(ent)
    return gt, prompt

In [24]:
def generate_prompts(triple_df, ent_id_lst, lang_lst, prop_lst):
    prompt = []
    for ent in ent_id_lst:
        ent_names = triple_df.loc[ent,'labels']
        for lang in lang_lst:
            #check if ent label exists in lang
            if lang in ent_names.keys():
                name = ent_names[lang]
            else:
                name = ent_names['en']
            for prop in prop_lst:
                if not pd.isnull(triple_df.loc[ent,'P19']):
                    pob_gt, pob_prompt = prompt_generate_pob(name,triple_df.loc[ent],lang)
                if not pd.isnull(triple_df.loc[ent,'P569']):
                    dob_gt, dob_prompt = prompt_generate_dob(name,triple_df.loc[ent],lang)
                if not pd.isnull(triple_df.loc[ent,'P106']):
                    occ_gt, occ_prompt = prompt_generate_occ(name,triple_df.loc[ent],lang)
                if not pd.isnull(triple_df.loc[ent,'P27']):
                    country_gt, country_prompt = prompt_generate_country(name,triple_df.loc[ent],lang)
            prompt.append({'ent_ID':ent, 'lang':lang, 'pob_ground_truth':pob_gt,'pob_prompt':pob_prompt,'dob_ground_truth':dob_gt,'dob_prompt':dob_prompt,'occ_ground_truth':occ_gt,'occ_prompt':occ_prompt,'country_ground_truth':country_gt,'country_prompt':country_prompt,'num_sitelinks':triple_df.loc[ent,'sitelinks']})      
    return prompt

In [25]:
file_names = {'ar':'Arabic','zh':'Chinese','en':'English','fr':'French','de':'German','ru':'Russian','it':'Italian','pl':'Polish','hi':'Hindi'}
lang_lst = ['ar', 'de', 'en', 'fr', 'ru', 'zh', 'it', 'pl','hi']
prop_lst = ['P19','P569','P106','P27']

result_df = pd.DataFrame(columns=['ent_ID','pob_ground_truth','pob_prompt','dob_ground_truth','dob_prompt','occ_ground_truth','occ_prompt','country_ground_truth','country_prompt','num_sitelinks'])

for lang, file in file_names.items():
    ground_truth_df = read_input(file, lang)
    ent_id_lst = ground_truth_df.index
    prompts = generate_prompts(ground_truth_df, ent_id_lst, lang_lst, prop_lst)
    
    result_df = pd.DataFrame(prompts)
    result_df = result_df.sort_values(['num_sitelinks'], ascending=False)
    result_df.to_json('data/dataset_v2/prompt_complete_entities/prompt_'+lang+'.json',indent=4, orient='records', force_ascii=False)


In [26]:
ent_ar = pd.read_json('data/dataset_v2/prompt_complete_entities/prompt_ar.json')


In [27]:
ent_ar.groupby('lang').count()

,ent_ID,pob_ground_truth,pob_prompt,dob_ground_truth,dob_prompt,occ_ground_truth,occ_prompt,country_ground_truth,country_prompt,num_sitelinks
lang,,,,,,,,,,
ar,514,514,514,514,514,514,514,514,514,514
de,514,514,514,514,514,514,514,514,514,514
en,514,514,514,514,514,514,514,514,514,514
fr,514,514,514,514,514,514,514,514,514,514
hi,514,514,514,514,514,514,514,514,514,514
it,514,514,514,514,514,514,514,514,514,514
pl,514,514,514,514,514,514,514,514,514,514
ru,514,514,514,514,514,514,514,514,514,514
zh,514,514,514,514,514,514,514,514,514,514
